# LLM Lab — QLoRA Fine-tuning Quickstart

**Hardware**: RTX 4060 Laptop (8GB VRAM) — QLoRA only  
**Strategy**: 4-bit base + LoRA adapters (r=16) → ~5-6GB VRAM for 7-8B models

---
## 0. Install dependencies (first run only)

In [ ]:
# Run once — comment out after first install
# !pip install unsloth transformers datasets trl peft accelerate bitsandbytes

## 1. Prepare a dataset

Format: JSONL with `instruction` and `output` fields.

In [ ]:
import json, os

os.makedirs('datasets', exist_ok=True)

# Example dataset — replace with your own
samples = [
    {"instruction": "What is the capital of France?", "output": "Paris."},
    {"instruction": "Write a haiku about GPUs.",
     "output": "Silicon dreams glow / Tensors cascade through the dark / VRAM overflows"},
    {"instruction": "Explain QLoRA in one sentence.",
     "output": "QLoRA fine-tunes a quantized 4-bit base model using low-rank adapters, fitting 7B+ models on 8GB VRAM."},
]

dataset_path = 'datasets/example.jsonl'
with open(dataset_path, 'w') as f:
    for s in samples:
        f.write(json.dumps(s) + '\n')

print(f'Wrote {len(samples)} examples to {dataset_path}')

## 2. Load base model in 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"  # change to your model
# Or use a local GGUF path by converting to safetensors first

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
print(f"Model loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.2f}GB")

## 3. Attach LoRA adapter

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Train

In [ ]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

def format_prompt(ex):
    return {"text": f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['output']}"}

import json
with open(dataset_path) as f:
    records = [json.loads(l) for l in f if l.strip()]
dataset = Dataset.from_list(records).map(format_prompt)

OUTPUT_DIR = "exports/my-adapter"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        output_dir="runs",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="paged_adamw_8bit",
        report_to="none",
    ),
)

trainer.train()

## 5. Save adapter

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

## 6. Export to GGUF

Run this from the repository root:

In [ ]:
# !python finetune/export_gguf.py \
#   --adapter finetune/exports/my-adapter \
#   --base    meta-llama/Meta-Llama-3-8B-Instruct \
#   --name    my-finetuned-llama3

print("Uncomment the command above and run it to export to GGUF.")
print("The GGUF will appear in models/shared/ and in the LLM Lab UI.")